[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week9_monitoring_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 9 -- Monitoring in Production (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Evidently continuous monitoring, Prometheus metrics, Grafana dashboard, retrospective drift analysis

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Deploy an Evidently AI continuous drift monitor running on each weekly data batch
- Instrument the FastAPI server with Prometheus metrics: request count, latency, prediction distribution
- Build a four-panel Grafana dashboard: approval rate, p99 latency, default probability distribution, drift status
- Run a retrospective analysis: at which quarter would this monitoring have detected the drift
- Connect the Evidently drift trigger to the Week 5 Airflow retraining DAG



---

## MONDAY -- "Evidently Continuous Monitoring"


Evidently as a continuous service: receives data batches, computes drift metrics, stores results with timestamps. Grafana reads from the workspace. PSI exceeding 0.25 turns the drift panel amber; exceeding 0.40 turns it red and fires an alert.

Pause and Predict -- how often should the drift monitor run? What is the trade-off between detection speed and the risk of noise-driven alerts?


### Exercise 9.1 -- Evidently time series

Run drift monitor on Q1-Q4. Log drift_share and top 3 PSI features per batch to MLflow. Plot drift_share over time.


In [ ]:
# Exercise 9.1: Evidently time series
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Evidently Continuous Monitoring --
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.ui.workspace import Workspace
import pandas as pd

workspace = Workspace.create('monitoring/evidently_workspace')
project   = workspace.create_project('creditdefault-monitoring')
ref_data  = pd.read_csv('datasets/creditdefault_q1.csv')

def run_drift_monitor(batch_path: str, batch_name: str) -> dict:
    current = pd.read_csv(batch_path)
    report  = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=ref_data, current_data=current)
    project.add_report(report)
    result = report.as_dict()['metrics'][0]['result']
    return {'batch': batch_name,
            'drifted': result['number_of_drifted_columns'],
            'total':   result['number_of_columns'],
            'drift_share': result['share_of_drifted_columns']}

for q, path in [('Q2','datasets/creditdefault_q2.csv'),
                ('Q3','datasets/creditdefault_q3.csv'),
                ('Q4','datasets/creditdefault_q4.csv')]:
    r = run_drift_monitor(path, q)
    print(f"{r['batch']}: {r['drifted']}/{r['total']} ({r['drift_share']:.1%})")


### Expected Output (illustrative — Evidently workspace)

```
Q2: 1/23 (4.3%)
Q3: 2/23 (8.7%)
Q4: 6/23 (26.1%)
```
Q4 is the first quarter to cross the 25% drift-share alert line — exactly the quarter Farida
manually noticed the problem in the book's narrative. The goal of this week's monitoring stack
is to have caught this automatically two quarters earlier, in Q2.


### Monday Morning Moment

*Slack -- Monday, 9:00am.*

**Temi Adeyemi:** Should model quality monitoring and infrastructure monitoring go in the same dashboard?

**Dr. Emeka Obi:** One dashboard. At 3am the on-call engineer looks at one screen.

**Temi Adeyemi:** Which panel should be largest?

**Dr. Emeka Obi:** The approval rate. Farida checks it daily. If she can read the dashboard herself, she does not need to call me.

**Sola Fashola:** Grafana dashboards must be in version control as JSON. Every threshold change is a git commit.




---

## TUESDAY -- "Prometheus Instrumentation"


Three model-specific metrics: prediction count by decision (APPROVE/REVIEW/DECLINE), prediction latency histogram, and default probability distribution. If the APPROVE fraction rises with no server errors, something changed in the model's behaviour.


### Exercise 9.2 -- Prometheus instrumentation

Add all three metrics. Verify at /metrics in Prometheus exposition format. Send 20 requests and confirm counters and histograms update.


In [ ]:
# Exercise 9.2: Prometheus instrumentation
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Prometheus Instrumentation --
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
from fastapi import Response
import time

PREDICTIONS = Counter('credit_predictions_total', 'Predictions by decision', ['decision'])
LATENCY     = Histogram('credit_prediction_latency_seconds', 'Latency',
                          buckets=[0.01, 0.05, 0.1, 0.2, 0.5, 1.0])
DEFAULT_P   = Histogram('credit_default_probability', 'Default probability distribution',
                          buckets=[i/10 for i in range(11)])

@app.post('/predict')
def predict(application: Application):
    t0   = time.time()
    X    = [[application.LIMIT_BAL, application.PAY_0, application.PAY_2,
             application.BILL_AMT1, application.PAY_AMT1]]
    prob = model.predict_proba(X)[0][1]
    decision = 'APPROVE' if prob<0.35 else 'REVIEW' if prob<0.60 else 'DECLINE'
    PREDICTIONS.labels(decision=decision).inc()
    LATENCY.observe(time.time() - t0)
    DEFAULT_P.observe(float(prob))
    return {'customer_id': application.customer_id,
            'default_probability': round(float(prob), 4), 'decision': decision}

@app.get('/metrics')
def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)


### Expected Output (illustrative — Prometheus `/metrics`)

```
$ curl http://localhost:8000/metrics | grep credit_predictions_total
credit_predictions_total{decision="APPROVE"} 812.0
credit_predictions_total{decision="REVIEW"} 143.0
credit_predictions_total{decision="DECLINE"} 45.0
```
Prometheus scrapes this endpoint on an interval (commonly every 15s) — the counters only go
up, so Grafana computes the *rate* of change (via `rate(...)` in PromQL) to get a meaningful
per-minute or per-hour figure, not the raw cumulative total.



---

## WEDNESDAY -- "Grafana Dashboard"


Four panels. Panel 1: approval rate trend (Farida's metric -- make it most prominent). Panel 2: p99 latency. Panel 3: default probability distribution. Panel 4: Evidently drift status. All four on one screen. One place to look at 3am.

Pause and Predict -- which panel should be largest? Which would Farida want prominent? Which is most relevant for the on-call engineer at 3am?


### Exercise 9.3 -- Grafana dashboard JSON

Build the four-panel dashboard. Set alert thresholds. Export as JSON and commit to the repository.


In [ ]:
# Exercise 9.3: Grafana dashboard JSON
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Grafana Dashboard --
# Grafana panels (PromQL):
panels = [
    {'title': 'Approval Rate (5-min rolling)',
     'promql': ('rate(credit_predictions_total{decision="APPROVE"}[5m]) /'
                ' rate(credit_predictions_total[5m])'),
     'thresholds': {'amber': 0.39, 'red': 0.44}},
    {'title': 'p99 Latency (ms)',
     'promql': 'histogram_quantile(0.99, rate(credit_prediction_latency_seconds_bucket[5m])) * 1000',
     'thresholds': {'amber': 150, 'red': 200}},
    {'title': 'Default Probability Distribution',
     'promql': 'rate(credit_default_probability_bucket[5m])'},
    {'title': 'Data Drift Status',
     'source': 'evidently',
     'thresholds': {'amber': 0.10, 'red': 0.25}},
]
print('Export as JSON from Grafana and commit to version control')


### Expected Output (illustrative)

```
Export as JSON from Grafana and commit to version control
```
The real deliverable here isn't the print statement — it's the dashboard JSON. Committing it
to git means the dashboard is versioned the same way the model and data are; a dashboard that
only lives in Grafana's database is one bad upgrade away from being unrecoverable.



---

## THURSDAY -- "Retrospective Detection Analysis"


If this monitoring had existed from January, when would it have detected the drift? Load each quarterly batch and run the drift monitor. The first quarter where drift_share exceeds 0.25 is when the alert would have fired.


### STOP AND TRACE

Before running:

LATENCY = Histogram('latency_seconds', 'Latency',
    buckets=[0.01, 0.05, 0.1, 0.2, 0.5, 1.0])

If 99% of requests complete in under 150ms, which bucket contains the p99?
If the p99 is 250ms, which bucket does histogram_quantile(0.99, ...) report?
Why this matters: histogram_quantile interpolates within the bucket containing the percentile. If your 200ms SLA sits between the 0.1 and 0.5 buckets, Grafana cannot distinguish 150ms from 499ms. Add a bucket at 0.200 for accurate SLA monitoring.


In [ ]:
# Exercise 9.4: STOP AND TRACE -- Prometheus histogram p99
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Retrospective Detection Analysis --
ref = pd.read_csv('datasets/creditdefault_q1.csv')
batches = [('Q2','datasets/creditdefault_q2.csv'),
           ('Q3','datasets/creditdefault_q3.csv'),
           ('Q4','datasets/creditdefault_q4.csv')]

first_alert = None
for name, path in batches:
    r = run_drift_monitor(path, name)
    status = 'ALERT' if r['drift_share'] > 0.25 else 'OK'
    print(f'{name}: {r["drifted"]}/{r["total"]} ({r["drift_share"]:.1%}) -- {status}')
    if status == 'ALERT' and first_alert is None:
        first_alert = name

print(f'\nMonitoring would have alerted during: {first_alert}')
print('Compare to: Farida flagged the problem in Q4 (two quarters later)')


### Expected Output (illustrative)

```
Q2: 1/23 (4.3%) -- OK
Q3: 2/23 (8.7%) -- OK
Q4: 6/23 (26.1%) -- ALERT

Monitoring would have alerted during: Q4
Compare to: Farida flagged the problem in Q4 (two quarters later)
```
Read that last comparison carefully: in this simulation, the monitoring and the human notice
in the *same* quarter. The lesson is about the four months of Q2-Q3 where automated monitoring
was silent and *should* have alerted earlier under a stricter threshold — worth testing what
threshold would have caught it in Q2 instead of Q4.



---

## FRIDAY -- "The Friday Build: Live Dashboard"


Send 50 requests to the deployed model. Verify in Grafana: approval rate panel updates within 30 seconds, p99 latency panel shows a value under 200ms, default probability histogram is populated. Run Evidently on Q4 data and verify the drift panel turns amber.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Live Dashboard --
import requests, random

for _ in range(50):
    payload = {'customer_id': random.randint(1000, 30000),
               'LIMIT_BAL':   random.choice([50000, 100000, 200000, 500000]),
               'PAY_0':       random.choice([-1, 0, 1, 2]),
               'PAY_2':       random.choice([-1, 0, 1]),
               'BILL_AMT1':   random.uniform(0, 200000),
               'PAY_AMT1':    random.uniform(0, 50000)}
    resp = requests.post('http://localhost/predict', json=payload)
    print(resp.json()['decision'], end=' ', flush=True)
print('\nCheck Grafana: grafana.dataflow.ng:3000')


### Expected Output (illustrative)

```
APPROVE APPROVE REVIEW APPROVE DECLINE APPROVE APPROVE REVIEW ...
Check Grafana: grafana.dataflow.ng:3000
```
50 synthetic requests should now be visible as fresh data points on all four Grafana panels
from Week 9's dashboard (Figure 9.1) — if the latency or drift panels don't move at all, check
that Prometheus is actually scraping this server's `/metrics` endpoint before assuming the
dashboard config is wrong.


### Friday Workplace Moment

*DataFlow -- Friday standup.*

**Dr. Emeka Obi:** Retrospective. When would the alert have fired?

**Temi Adeyemi:** Q2. LIMIT_BAL PSI exceeded 0.25 in Q2 data. Alert would have fired two quarters before Farida flagged the problem.

**Farida Usman:** Two quarters earlier. How many decisions is that?

**Temi Adeyemi:** Approximately 5,000 decisions in Q2 and Q3 that were assessed with an unreliable model.

**Dr. Emeka Obi:** Write that in the monitoring design document. That is the business case.



## YOUR WEEK 9 CHECKLIST

Check each item before moving on.

- [ ] Evidently drift monitor runs on each new data batch and results are visible in Grafana.
- [ ] FastAPI exposes Prometheus metrics at /metrics including prediction distribution.
- [ ] Grafana dashboard has four panels and is committed to version control as JSON.
- [ ] Retrospective analysis identifies the earliest detection date and quantifies the gap.
- [ ] 50 requests populate all four Grafana panels within 60 seconds.


## Challenge Task

> **Core path:** Implement anomaly detection on the default probability mean: alert if the current hour falls more than 2 standard deviations from a 24-hour rolling mean.

> **Advanced path:** Build a weekly labelled evaluation job: sample 200 predictions, retrieve actual outcomes, compute AUC, log to Grafana.


## Common Mistakes

**Monitoring only infrastructure:** A model with healthy latency and zero errors can be systematically wrong. Model quality monitoring is not optional.

**Alert thresholds too sensitive:** An alert firing on every 1% fluctuation will be suppressed within a week.

**Grafana dashboard not version-controlled:** A dashboard changed in the UI can be accidentally deleted. Export as JSON and commit every change.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)